# 🔄 Backpropagation — Notes + Interview
---
> **Simple English** | **Interview Ready**

## 📌 What is Backpropagation? (Simple English)
- Backprop = how the network **learns from its mistakes**
- After forward pass gives prediction → compute loss (how wrong?)
- Backprop sends error signal **backward** through network
- Calculates: how much did each weight **contribute to the error**?
- Then optimizer updates weights to reduce that error
- Uses the **chain rule** from calculus

## 🔑 Full Training Loop
```
1. Forward Pass   → compute ŷ (prediction)
2. Compute Loss   → Loss(ŷ, y)
3. Backward Pass  → compute dLoss/dWeight (gradient)
4. Update Weights → w = w - learning_rate × gradient
5. Repeat for all batches / epochs
```

## 🔑 Chain Rule (Simple)
```
If Loss depends on a, and a depends on z, and z depends on W:
dLoss/dW = dLoss/da  ×  da/dz  ×  dz/dW
         = multiply gradients layer by layer going BACKWARD
```

In [ ]:
import numpy as np

np.random.seed(42)
X = np.array([[1.0,2.0],[3.0,1.0],[0.5,1.5]])
y = np.array([[1.0],[0.0],[1.0]])

W1 = np.random.randn(2,3)*0.1; b1 = np.zeros((1,3))
W2 = np.random.randn(3,1)*0.1; b2 = np.zeros((1,1))
lr = 0.1

sigmoid = lambda z: 1/(1+np.exp(-z))
sig_d   = lambda a: a*(1-a)  # sigmoid derivative

# FORWARD PASS
z1=X.dot(W1)+b1; a1=sigmoid(z1)
z2=a1.dot(W2)+b2; a2=sigmoid(z2)
loss = np.mean((y-a2)**2)
print(f"Initial Loss: {loss:.4f} | Preds: {a2.flatten().round(3)}")

In [ ]:
# BACKWARD PASS — compute gradients using chain rule
n = len(X)

# Output layer
dL_da2 = -2*(y-a2)/n
dL_dz2 = dL_da2 * sig_d(a2)
dL_dW2 = a1.T.dot(dL_dz2)
dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)

# Hidden layer (chain rule goes backward)
dL_da1 = dL_dz2.dot(W2.T)
dL_dz1 = dL_da1 * sig_d(a1)
dL_dW1 = X.T.dot(dL_dz1)
dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

print("Gradients computed:")
print(f"dL/dW2: {dL_dW2.shape} | dL/dW1: {dL_dW1.shape}")

# UPDATE WEIGHTS
W2 -= lr*dL_dW2; b2 -= lr*dL_db2
W1 -= lr*dL_dW1; b1 -= lr*dL_db1

# New loss after 1 backprop step
z1=X.dot(W1)+b1; a1=sigmoid(z1)
z2=a1.dot(W2)+b2; a2=sigmoid(z2)
loss_new = np.mean((y-a2)**2)
print(f"\nLoss: {loss:.4f} → {loss_new:.4f}  ✅ Reduced!")

In [ ]:
# TensorFlow does backprop automatically with GradientTape
import tensorflow as tf

W = tf.Variable([[0.5, -0.3],[0.2, 0.8]], dtype=tf.float32)
x_in  = tf.constant([[1.0, 2.0]])
y_true = tf.constant([[1.0, 0.0]])

with tf.GradientTape() as tape:
    y_pred = tf.nn.sigmoid(tf.matmul(x_in, W))
    loss   = tf.reduce_mean((y_true - y_pred)**2)

gradients = tape.gradient(loss, W)
print("Auto-computed gradients (dLoss/dW):")
print(gradients.numpy().round(6))
print("\n✅ GradientTape does all the chain rule math automatically!")

## 🗣️ Interview Questions & Answers

**Q: What is backpropagation?**
> Algorithm to compute gradient of loss w.r.t. each weight by applying chain rule backward through the network. These gradients are then used by the optimizer to update weights.

**Q: What is the chain rule in backprop?**
> If f(g(x)), then df/dx = df/dg × dg/dx. In networks: multiply gradients layer-by-layer going backward to find how much each weight contributed to total loss.

**Q: What is the vanishing gradient problem?**
> In deep networks with sigmoid/tanh, gradients shrink exponentially going backward. Early layers learn very slowly or stop learning. Solutions: ReLU activation, batch normalization, residual connections (ResNet).

**Q: What is the exploding gradient problem?**
> Opposite — gradients grow very large, causing unstable weight updates. Solutions: gradient clipping, He initialization, batch normalization.